# DeBERTa-v3-xsmall deployment preflight

This notebook is the readable Phase 4 entry point. It lives in `model/DeBERTa-v3-xsmall/preflight/`; the Model V2 training notebook is the one at the folder root. It does not train Model V2. The temporary 67-label classifier head is random and exists only to validate deployment compatibility.

## Deployment contract

The preflight pins the upstream revision, validates exact tokenizer offsets and offline save/reload, exports dynamic batch and sequence axes at opset 17, applies ONNX Runtime dynamic QInt8 quantization, and benchmarks 75, 221, 289 and 512 tokens. Large generated artifacts are ignored.

In [ ]:
import json
import sys
from pathlib import Path

# Discover the repository from any cwd inside it, then import the adjacent module.
start = Path.cwd().resolve()
repo_root = next((root for root in (start, *start.parents)
                  if (root / "FullPipeline_V3").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Run this notebook from inside the OnKith repository.")
HERE = repo_root / "model" / "DeBERTa-v3-xsmall" / "preflight"
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from preflight_deberta_v3_xsmall import RESULTS_PATH, concise_summary, run_preflight

## Run the full preflight

Run this only when regenerating ignored weights and ONNX graphs. On Windows, start Jupyter with `PYTHONUTF8=1` so the modern exporter can print its progress markers.

In [ ]:
# results = run_preflight()
# concise_summary(results)

## Inspect the saved compact evidence

This reads the committed report without redownloading or exporting anything.

In [ ]:
saved_results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
concise_summary(saved_results)

## Interpretation

Passing export and runtime checks proves deployability only. It does not establish PII accuracy because the classification head is untrained. Raspberry Pi latency and co-resident memory remain separate hardware gates.